In [1]:
import gradio as gr
import numpy as np
import pandas as pd
import joblib

# 1. Load the exported pickle files
try:
    kmeans = joblib.load("/content/kmeans_anomaly_model.pkl")
    scaler = joblib.load("/content/data_scaler.pkl")
    print("🚀 Model and Scaler loaded successfully!")
except FileNotFoundError:
    print("❌ Error: Could not find 'kmeans_anomaly_model.pkl' or 'data_scaler.pkl'.")
    print("Please make sure you uploaded both files to this notebook environment.")

# 2. Define the prediction wrapper function
def check_for_anomaly(ambient_value, hour_of_day):
    try:
        # Create a DataFrame utilizing the exact feature names your scaler expects ('value', 'hour')
        input_df = pd.DataFrame([[float(ambient_value), int(hour_of_day)]], columns=['value', 'hour'])

        # Standardize the data using your original training dataset's metrics
        scaled_input = scaler.transform(input_df)

        # Find the index of closest cluster center
        cluster_assignment = kmeans.predict(scaled_input)[0]

        # Calculate Euclidean distance between the entry and its assigned center
        center_coords = kmeans.cluster_centers_[cluster_assignment]
        distance = np.linalg.norm(scaled_input - center_coords)

        # --- THRESHOLD CRITERIA ---
        # Modify this limit to align with your notebook's anomaly threshold (e.g., 95th percentile)
        MAX_NORMAL_THRESHOLD = 2.5
        # --------------------------

        is_anomaly = distance > MAX_NORMAL_THRESHOLD
        verdict = "🚨 SYSTEM ANOMALY DETECTED" if is_anomaly else "✅ SYSTEM OPERATING NORMALLY"

        # Generate clean human-readable metrics markdown
        report = (
            f"### Status: {verdict}\n\n"
            f"**Assigned Cluster Center ID:** {cluster_assignment}\n"
            f"**Calculated Data Distance:** {distance:.4f}\n"
            f"**Allowed Variance Threshold:** {MAX_NORMAL_THRESHOLD:.4f}\n\n"
            f"*Interpretation: {'The variance falls significantly outside regular bounds for this time of day.' if is_anomaly else 'The variance falls inside the expected normal bounds.'}*"
        )
        return report

    except Exception as e:
        return f"### ❌ Operational Error\nAn issue occurred processing input calculations: `{str(e)}`"

# 3. Design the Graphical Interface Dashboard
interface = gr.Interface(
    fn=check_for_anomaly,
    inputs=[
        gr.Number(
            label="Ambient Telemetry Value (e.g. Temperature)",
            value=71.2,
            info="Enter the reading recorded from the machine or application log."
        ),
        gr.Slider(
            minimum=0,
            maximum=23,
            step=1,
            label="Hour of the Day (0-23)",
            value=12,
            info="Slide to select the exact hour the log entry happened."
        )
    ],
    outputs=gr.Markdown(label="Real-time Evaluation Output"),
    title="🔍 KMeans Cloudwatch & System Anomaly Detector",
    description="This operational dashboard maps real-time ambient values against an entry's hour window using your pre-trained clustering layout to flag systemic anomalies.",
    theme="glass",
    allow_flagging="never"
)

# 4. Fire up the application locally inside your notebook
if __name__ == "__main__":
    interface.launch()

🚀 Model and Scaler loaded successfully!


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a1759653f5a451da8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
